# Bayesian Visits Planner: How Many Cafes to Pin Down CVR

**Method:** Beta-Binomial conjugate model — plan how many visits are needed until the posterior credible interval on the true CVR is as narrow as you want.

何軒回れば「本当のCVR」がだいたい分かるか？
---

$$\text{Posterior after } n \text{ visits: } \theta \,|\, x \sim \text{Beta}(\alpha_0 + x, \; \beta_0 + n - x), \quad x \approx p_{\text{guess}} \cdot n$$

$$\text{Find smallest } n \text{ such that the } 95\% \text{ credible interval half-width} \le d$$

$$\text{Rule of thumb: } n \approx \frac{z^2 \, p_{\text{guess}}(1 - p_{\text{guess}})}{d^2}$$

---

| Symbol | Meaning |
|--------|---------|
| $\theta$ | True conversion rate (unknown parameter) |
| $p_{\text{guess}}$ | Assumed CVR — pure guess, used only to plan (no data yet) |
| $d$ | Desired precision: half-width of the credible interval (± percentage points) |
| $n$ | Number of cafes visited (trials) |
| $x$ | Number of conversions expected under the guess |
| $\alpha_0, \beta_0$ | Prior hyperparameters (Beta(1,1) when the guess carries no evidence weight) |

## Data Input

No observed data yet — enter your planning assumptions below.

In [1]:
import json

config = json.load(open("../test_config.json"))
credible_threshold = 1 - config["design"]["alpha"]  # alpha=0.05 → 0.95

# ==========================================
#  PLANNING ASSUMPTIONS — pure guess, no data yet
# ==========================================
prior_cvr = 0.20    # assumed CVR (pure guess: ~1 in 5 cafes converts)
precision = 0.05    # want CVR pinned down to ± this (5 percentage points)
prior_n   = 0       # prior evidence weight (0 = guess carries no data weight)
# ==========================================

prior_alpha = 1 + prior_cvr * prior_n
prior_beta  = 1 + (1 - prior_cvr) * prior_n

In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta
from scipy.optimize import brentq

def posterior_halfwidth(n, p, prior_alpha, prior_beta, credible_threshold=0.95):
    """Expected credible interval half-width after n visits, assuming observed CVR ≈ p."""
    tail = (1 - credible_threshold) / 2
    a = prior_alpha + p * n
    b = prior_beta + (1 - p) * n
    return (beta.ppf(1 - tail, a, b) - beta.ppf(tail, a, b)) / 2

def required_visits(p, precision, prior_alpha, prior_beta, credible_threshold=0.95):
    if posterior_halfwidth(1, p, prior_alpha, prior_beta, credible_threshold) <= precision:
        return 1
    n = brentq(lambda n: posterior_halfwidth(n, p, prior_alpha, prior_beta, credible_threshold) - precision,
               1, 10_000_000)
    return int(np.ceil(n))

def plan_bayesian_precision(p, precision, prior_alpha, prior_beta, credible_threshold=0.95):
    thresh_pct = f"{credible_threshold:.0%}"
    req_n = required_visits(p, precision, prior_alpha, prior_beta, credible_threshold)
    hw = posterior_halfwidth(req_n, p, prior_alpha, prior_beta, credible_threshold)

    n_arr = np.linspace(10, req_n * 3, 400)
    w_arr = np.array([posterior_halfwidth(n, p, prior_alpha, prior_beta, credible_threshold) for n in n_arr])

    # Sweet spot (knee): point farthest from the chord joining the curve's endpoints,
    # in normalized coordinates (Kneedle). The tangent there is parallel to the chord.
    Xn = (n_arr - n_arr[0]) / (n_arr[-1] - n_arr[0])
    Yn = (w_arr - w_arr[-1]) / (w_arr[0] - w_arr[-1])
    k = int(np.argmax(np.abs(Xn + Yn - 1)))
    knee_n, knee_w = n_arr[k], w_arr[k]
    m = -(w_arr[0] - w_arr[-1]) / (n_arr[-1] - n_arr[0])              # tangent slope = chord slope
    tan_x0, tan_x1 = n_arr[0], min(n_arr[-1], knee_n + knee_w / -m)   # clip where the line reaches zero
    tan_y0, tan_y1 = knee_w + m * (tan_x0 - knee_n), knee_w + m * (tan_x1 - knee_n)

    print(f"--- ベイズ推定 訪問数プランナー ---")
    print(f"手法 : ベイズ推定（ベータ二項分布モデル）")
    print(f"KPI  : コンバージョン率（CVR）")
    print(f"想定CVR : {p:.1%}（純粋な推測・データなし）  |  Prior : Beta({prior_alpha:g}, {prior_beta:g})")
    print(f"目標精度 : ±{precision:.1%}（{thresh_pct}信用区間の半幅）")
    print(f"-" * 50)
    print(f"必要訪問数 : {req_n:,} 軒")
    print(f"→ {req_n:,}軒訪問後、真のCVRはおよそ {max(p - hw, 0):.1%} 〜 {p + hw:.1%} に絞り込める（{thresh_pct}）")
    print(f"-" * 50)
    print(f"スイートスポット（収穫逓減の目安）: 約 {knee_n:,.0f} 軒（±{knee_w:.1%}）")
    print(f"→ ここを超えると1軒あたりの精度向上が目に見えて鈍る")
    print(f"-" * 50)
    print(f"【精度別の必要訪問数】")
    for d in [0.10, 0.05, 0.03, 0.02, 0.01]:
        n_d = required_visits(p, d, prior_alpha, prior_beta, credible_threshold)
        marker = "  ← あなたのプラン" if abs(d - precision) < 1e-9 else ""
        print(f"  ±{d:.1%} : {n_d:>7,} 軒{marker}")
    print(f"-" * 50)

    # --- Plot ---
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=n_arr, y=w_arr, mode='lines', name='CI half-width vs N', line=dict(color='#1f77b4', width=3)))
    fig.add_trace(go.Scatter(x=[n_arr[0], n_arr[-1]], y=[w_arr[0], w_arr[-1]], mode='lines', name='Endpoint chord',
        line=dict(color='#c3c2b7', width=1, dash='dot')))
    fig.add_trace(go.Scatter(x=[tan_x0, tan_x1], y=[tan_y0, tan_y1], mode='lines', name='Tangent at sweet spot',
        line=dict(color='#ff7f0e', width=2, dash='dash')))
    fig.add_trace(go.Scatter(x=[knee_n], y=[knee_w], mode='markers', name=f'Sweet spot (N≈{knee_n:,.0f})',
        marker=dict(color='#ff7f0e', size=13, symbol='star')))
    fig.add_trace(go.Scatter(x=[req_n], y=[hw], mode='markers', name=f'Your Plan (N={req_n:,})',
        marker=dict(color='red', size=12, symbol='diamond')))
    fig.add_hline(y=precision, line_dash="dot", line_color="gray",
        annotation_text=f"Target precision: ±{precision:.1%}", annotation_position="top right")
    fig.add_annotation(x=knee_n, y=knee_w, text=f"<b>Sweet spot ≈ {knee_n:,.0f} cafes</b><br>±{knee_w:.1%}",
        ax=70, ay=-55, font=dict(size=12, color="#ff7f0e"), arrowcolor="#ff7f0e")
    fig.add_annotation(x=req_n, y=hw, text=f"<b>{req_n:,} cafes</b><br>±{hw:.1%}",
        ax=40, ay=-40, font=dict(size=12, color="red"), arrowcolor="red")

    fig.update_layout(
        title=dict(text=f"<b>Bayesian Visits Planner</b><br>Assumed CVR: {p:.1%} (pure guess) | {thresh_pct} CI half-width vs cafes visited", font=dict(size=20)),
        xaxis_title="Cafes Visited (N)", yaxis_title=f"{thresh_pct} CI Half-Width (±)",
        xaxis=dict(showgrid=True, gridcolor='#eee'), yaxis=dict(tickformat=".1%", showgrid=True, gridcolor='#eee'),
        template="plotly_white", legend=dict(y=0.99, x=0.99, bgcolor="rgba(255,255,255,0.8)"), plot_bgcolor='rgba(0,0,0,0)')
    fig.show()
    return req_n

In [3]:
plan_bayesian_precision(p=prior_cvr, precision=precision,
    prior_alpha=prior_alpha, prior_beta=prior_beta,
    credible_threshold=credible_threshold)

--- ベイズ推定 訪問数プランナー ---
手法 : ベイズ推定（ベータ二項分布モデル）
KPI  : コンバージョン率（CVR）
想定CVR : 20.0%（純粋な推測・データなし）  |  Prior : Beta(1, 1)
目標精度 : ±5.0%（95%信用区間の半幅）
--------------------------------------------------
必要訪問数 : 245 軒
→ 245軒訪問後、真のCVRはおよそ 15.0% 〜 25.0% に絞り込める（95%）
--------------------------------------------------
スイートスポット（収穫逓減の目安）: 約 124 軒（±7.0%）
→ ここを超えると1軒あたりの精度向上が目に見えて鈍る
--------------------------------------------------
【精度別の必要訪問数】
  ±10.0% :      60 軒
  ±5.0% :     245 軒  ← あなたのプラン
  ±3.0% :     682 軒
  ±2.0% :   1,536 軒
  ±1.0% :   6,145 軒
--------------------------------------------------


245

## Interactive Explorer

Same tool as the hosted artifact, embedded in this notebook. Drag the sliders — the Beta curve, credible band, and quantiles update live. Everything runs in your browser; no kernel needed once the cell output is saved. (If the sliders ever don't respond after reopening, just run this cell once.)

In [4]:
from IPython.display import HTML

explorer_html = """<title>CVR Beta Explorer</title>
<style>
  :root {
    color-scheme: light;
    --plane: #f9f9f7;
    --surface: #fcfcfb;
    --ink: #0b0b0b;
    --ink-2: #52514e;
    --muted: #898781;
    --grid: #e1e0d9;
    --axis: #c3c2b7;
    --series: #2a78d6;
    --band: rgba(42, 120, 214, 0.16);
    --border: rgba(11, 11, 11, 0.10);
  }
  @media (prefers-color-scheme: dark) {
    :root:where(:not([data-theme="light"])) {
      color-scheme: dark;
      --plane: #0d0d0d;
      --surface: #1a1a19;
      --ink: #ffffff;
      --ink-2: #c3c2b7;
      --muted: #898781;
      --grid: #2c2c2a;
      --axis: #383835;
      --series: #3987e5;
      --band: rgba(57, 135, 229, 0.22);
      --border: rgba(255, 255, 255, 0.10);
    }
  }
  :root[data-theme="dark"] {
    color-scheme: dark;
    --plane: #0d0d0d;
    --surface: #1a1a19;
    --ink: #ffffff;
    --ink-2: #c3c2b7;
    --muted: #898781;
    --grid: #2c2c2a;
    --axis: #383835;
    --series: #3987e5;
    --band: rgba(57, 135, 229, 0.22);
    --border: rgba(255, 255, 255, 0.10);
  }

  body {
    background: var(--plane);
    color: var(--ink);
    font-family: system-ui, -apple-system, "Segoe UI", sans-serif;
    line-height: 1.45;
    margin: 0;
    padding: 28px 20px 48px;
  }
  .wrap { max-width: 1020px; margin: 0 auto; display: flex; flex-direction: column; gap: 18px; }

  header h1 { font-size: 1.35rem; margin: 0 0 4px; letter-spacing: -0.01em; }
  header p { margin: 0; color: var(--ink-2); font-size: 0.92rem; max-width: 62ch; }

  .card {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 6px;
    padding: 16px 18px;
  }

  /* controls */
  .controls { display: flex; flex-wrap: wrap; gap: 18px 28px; align-items: flex-end; }
  .ctl { display: flex; flex-direction: column; gap: 6px; min-width: 180px; flex: 1 1 180px; }
  .ctl-head { display: flex; justify-content: space-between; align-items: baseline; gap: 10px; }
  .ctl-head label { font-size: 0.72rem; text-transform: uppercase; letter-spacing: 0.06em; color: var(--muted); }
  .ctl-head output { font-size: 0.9rem; font-weight: 600; font-variant-numeric: tabular-nums; }
  input[type="range"] { width: 100%; accent-color: var(--series); margin: 0; }
  .seg { display: flex; border: 1px solid var(--border); border-radius: 6px; overflow: hidden; width: max-content; }
  .seg button {
    font: inherit; font-size: 0.82rem; padding: 4px 12px; border: 0; cursor: pointer;
    background: transparent; color: var(--ink-2); font-variant-numeric: tabular-nums;
  }
  .seg button + button { border-left: 1px solid var(--border); }
  .seg button[aria-pressed="true"] { background: var(--series); color: #fff; font-weight: 600; }
  .seg button:focus-visible { outline: 2px solid var(--series); outline-offset: -2px; }
  .ctl--fixed { flex: 0 0 auto; min-width: 0; }

  /* chart */
  .chart-card { position: relative; padding: 14px 10px 6px; }
  svg { display: block; width: 100%; height: auto; }
  .tooltip {
    position: absolute; pointer-events: none; display: none;
    background: var(--surface); border: 1px solid var(--border); border-radius: 6px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.12);
    padding: 6px 10px; font-size: 0.8rem; color: var(--ink-2); white-space: nowrap;
    font-variant-numeric: tabular-nums;
  }
  .tooltip strong { color: var(--ink); }

  /* readouts */
  .tiles { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; }
  @media (max-width: 720px) { .tiles { grid-template-columns: repeat(2, 1fr); } }
  .tile { display: flex; flex-direction: column; gap: 2px; }
  .tile .k { font-size: 0.72rem; text-transform: uppercase; letter-spacing: 0.06em; color: var(--muted); display: flex; align-items: center; gap: 6px; }
  .tile .k .dot { width: 8px; height: 8px; border-radius: 2px; background: var(--series); }
  .tile .v { font-size: 1.45rem; font-weight: 650; letter-spacing: -0.01em; }
  .tile .s { font-size: 0.78rem; color: var(--muted); }
  .tile--hero .v { font-size: 1.9rem; }

  /* table */
  table { border-collapse: collapse; width: 100%; font-size: 0.86rem; }
  caption { text-align: left; font-size: 0.72rem; text-transform: uppercase; letter-spacing: 0.06em; color: var(--muted); padding-bottom: 8px; }
  th, td { padding: 5px 10px; text-align: right; font-variant-numeric: tabular-nums; }
  th { color: var(--muted); font-weight: 500; border-bottom: 1px solid var(--grid); }
  td { color: var(--ink-2); }
  td:first-child, th:first-child { text-align: left; }
  tr + tr td { border-top: 1px solid var(--grid); }

  .foot { color: var(--muted); font-size: 0.78rem; max-width: 78ch; }
  .foot code { font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 0.95em; }
</style>

<div class="wrap">
  <header>
    <h1>CVR Beta Explorer</h1>
    <p>The curve is your belief about the true conversion rate of café visits. Drag the sliders and watch the plausible range narrow — the shaded band is where the true CVR sits with the chosen credibility.</p>
  </header>

  <section class="card">
    <div class="controls">
      <div class="ctl">
        <div class="ctl-head"><label for="p">Assumed CVR (your guess)</label><output id="pOut">20.0%</output></div>
        <input type="range" id="p" min="1" max="60" step="0.5" value="20">
      </div>
      <div class="ctl">
        <div class="ctl-head"><label for="n">Cafés visited</label><output id="nOut">245</output></div>
        <input type="range" id="n" min="0" max="2000" step="5" value="245">
      </div>
      <div class="ctl">
        <div class="ctl-head"><label for="w">Prior weight (pseudo-visits)</label><output id="wOut">0</output></div>
        <input type="range" id="w" min="0" max="200" step="5" value="0">
      </div>
      <div class="ctl ctl--fixed">
        <div class="ctl-head"><label>Credible level</label></div>
        <div class="seg" id="level" role="group" aria-label="Credible level">
          <button aria-pressed="false" data-v="0.80">80%</button>
          <button aria-pressed="false" data-v="0.90">90%</button>
          <button aria-pressed="true"  data-v="0.95">95%</button>
          <button aria-pressed="false" data-v="0.99">99%</button>
        </div>
      </div>
      <div class="ctl ctl--fixed">
        <div class="ctl-head"><label>X-axis</label></div>
        <div class="seg" id="zoom" role="group" aria-label="X axis range">
          <button aria-pressed="true"  data-v="full">0–100%</button>
          <button aria-pressed="false" data-v="fit">Fit curve</button>
        </div>
      </div>
    </div>
  </section>

  <section class="card chart-card">
    <svg id="chart" viewBox="0 0 960 400" role="img" aria-label="Beta distribution of the true conversion rate"></svg>
    <div class="tooltip" id="tip"></div>
  </section>

  <section class="card tiles">
    <div class="tile tile--hero">
      <span class="k"><span class="dot"></span>Interval width</span>
      <span class="v" id="tWidth">—</span>
      <span class="s">how tightly you'd know the CVR</span>
    </div>
    <div class="tile">
      <span class="k">Credible interval</span>
      <span class="v" id="tCI">—</span>
      <span class="s" id="tCIlvl">95% credible</span>
    </div>
    <div class="tile">
      <span class="k">Mean CVR</span>
      <span class="v" id="tMean">—</span>
      <span class="s">posterior expectation</span>
    </div>
    <div class="tile">
      <span class="k">Distribution</span>
      <span class="v" id="tBeta">—</span>
      <span class="s">Beta(α, β)</span>
    </div>
  </section>

  <section class="card">
    <table id="qtable">
      <caption>Quantiles of the true CVR</caption>
      <thead><tr><th scope="col">Percentile</th><th scope="col">2.5%</th><th scope="col">10%</th><th scope="col">25%</th><th scope="col">50%</th><th scope="col">75%</th><th scope="col">90%</th><th scope="col">97.5%</th></tr></thead>
      <tbody><tr><td>CVR</td><td>—</td><td>—</td><td>—</td><td>—</td><td>—</td><td>—</td><td>—</td></tr></tbody>
    </table>
  </section>

  <p class="foot">Model: CVR ~ <code>Beta(1 + p·(n + w), 1 + (1−p)·(n + w))</code> — a Beta-Binomial posterior that assumes conversions come in at your guessed rate <code>p</code>. <em>Cafés visited</em> adds real-data weight; <em>prior weight</em> is how many visits your gut feeling is worth (0 = pure guess, flat start). The vertical density scale is relative and carries no meaning — only the position and width of the curve matter.</p>
</div>

<script>
(() => {
  'use strict';

  // ---------- Beta math ----------
  const LG_C = [676.5203681218851, -1259.1392167224028, 771.32342877765313,
    -176.61502916214059, 12.507343278686905, -0.13857109526572012,
    9.9843695780195716e-6, 1.5056327351493116e-7];
  function logGamma(z) {
    if (z < 0.5) return Math.log(Math.PI / Math.sin(Math.PI * z)) - logGamma(1 - z);
    z -= 1;
    let x = 0.99999999999980993;
    for (let i = 0; i < 8; i++) x += LG_C[i] / (z + i + 1);
    const t = z + 7.5;
    return 0.5 * Math.log(2 * Math.PI) + (z + 0.5) * Math.log(t) - t + Math.log(x);
  }
  function makeDist(a, b) {
    const logB = logGamma(a) + logGamma(b) - logGamma(a + b);
    const pdf = x => {
      if (x <= 0) return a === 1 ? Math.exp(-logB) * Math.pow(1 - x, b - 1) : 0;
      if (x >= 1) return b === 1 ? Math.exp(-logB) * Math.pow(x, a - 1) : 0;
      return Math.exp((a - 1) * Math.log(x) + (b - 1) * Math.log(1 - x) - logB);
    };
    // dense grid CDF (trapezoid), for quantiles + hover cumulative
    const N = 4000, xs = new Float64Array(N + 1), cdf = new Float64Array(N + 1);
    let prev = pdf(0), acc = 0;
    xs[0] = 0; cdf[0] = 0;
    for (let i = 1; i <= N; i++) {
      const x = i / N, f = pdf(x);
      acc += (prev + f) / (2 * N);
      xs[i] = x; cdf[i] = acc; prev = f;
    }
    for (let i = 0; i <= N; i++) cdf[i] /= acc;   // normalize numeric drift
    const quantile = q => {
      let lo = 0, hi = N;
      while (hi - lo > 1) { const m = (lo + hi) >> 1; (cdf[m] < q) ? lo = m : hi = m; }
      const span = cdf[hi] - cdf[lo];
      return xs[lo] + (span > 0 ? (q - cdf[lo]) / span : 0) / N;
    };
    const cdfAt = x => {
      const t = Math.min(Math.max(x, 0), 1) * N, i = Math.floor(t);
      return i >= N ? 1 : cdf[i] + (cdf[i + 1] - cdf[i]) * (t - i);
    };
    const mean = a / (a + b);
    const sd = Math.sqrt(a * b / ((a + b) * (a + b) * (a + b + 1)));
    return { pdf, quantile, cdfAt, mean, sd, a, b };
  }

  // ---------- state ----------
  const $ = id => document.getElementById(id);
  const state = { p: 0.20, n: 245, w: 0, level: 0.95, zoom: 'full' };
  let dist = null, view = null;   // view: {x0, x1, yMax, plot rect}

  const fmtPct  = (x, d = 1) => (x * 100).toFixed(d) + '%';
  const fmtNum  = x => Number.isInteger(x) ? x.toLocaleString('en-US') : x.toFixed(1);

  // ---------- chart ----------
  const SVG_NS = 'http://www.w3.org/2000/svg';
  const W = 960, H = 400, M = { l: 14, r: 14, t: 26, b: 44 };
  const PW = W - M.l - M.r, PH = H - M.t - M.b;
  const svg = $('chart'), tip = $('tip');
  const el = (tag, attrs) => {
    const e = document.createElementNS(SVG_NS, tag);
    for (const k in attrs) e.setAttribute(k, attrs[k]);
    return e;
  };
  function niceStep(range) {
    const raw = range / 8, mag = Math.pow(10, Math.floor(Math.log10(raw)));
    for (const m of [1, 2, 5, 10]) if (raw <= m * mag) return m * mag;
    return 10 * mag;
  }

  function render() {
    const { p, n, w, level } = state;
    const a = 1 + p * (n + w), b = 1 + (1 - p) * (n + w);
    dist = makeDist(a, b);
    const tail = (1 - level) / 2;
    const qLo = dist.quantile(tail), qHi = dist.quantile(1 - tail);
    const half = (qHi - qLo) / 2;

    // domain
    let x0 = 0, x1 = 1;
    if (state.zoom === 'fit') {
      x0 = Math.max(0, dist.mean - 6 * dist.sd);
      x1 = Math.min(1, dist.mean + 6 * dist.sd);
      if (x1 - x0 < 0.01) { const c = (x0 + x1) / 2; x0 = Math.max(0, c - 0.005); x1 = Math.min(1, c + 0.005); }
    }
    const X = v => M.l + (v - x0) / (x1 - x0) * PW;

    // y scale from visible max
    let yMax = 0;
    for (let i = 0; i <= 400; i++) yMax = Math.max(yMax, dist.pdf(x0 + (x1 - x0) * i / 400));
    yMax *= 1.08;
    const Y = v => M.t + PH - v / yMax * PH;

    svg.textContent = '';

    // gridlines + x ticks
    const stepPct = niceStep((x1 - x0) * 100);
    const g = el('g', {});
    for (let t = Math.ceil(x0 * 100 / stepPct) * stepPct; t <= x1 * 100 + 1e-9; t += stepPct) {
      const x = X(t / 100);
      g.appendChild(el('line', { x1: x, y1: M.t, x2: x, y2: M.t + PH, stroke: 'var(--grid)', 'stroke-width': 1 }));
      const lbl = el('text', { x, y: M.t + PH + 18, 'text-anchor': 'middle', 'font-size': 12, fill: 'var(--muted)' });
      lbl.textContent = t.toFixed(stepPct < 1 ? 1 : 0) + '%';
      g.appendChild(lbl);
    }
    svg.appendChild(g);
    svg.appendChild(el('line', { x1: M.l, y1: M.t + PH, x2: M.l + PW, y2: M.t + PH, stroke: 'var(--axis)', 'stroke-width': 1 }));

    // paths
    const PTS = 600;
    let curve = '', band = '';
    for (let i = 0; i <= PTS; i++) {
      const x = x0 + (x1 - x0) * i / PTS;
      const pt = `${X(x).toFixed(2)},${Y(dist.pdf(x)).toFixed(2)}`;
      curve += (i ? ' L' : 'M') + pt;
      if (x >= qLo && x <= qHi) band += (band ? ' L' : `M${X(qLo).toFixed(2)},${(M.t + PH).toFixed(2)} L`) + pt;
    }
    if (band) band += ` L${X(qHi).toFixed(2)},${(M.t + PH).toFixed(2)} Z`;
    if (band) svg.appendChild(el('path', { d: band, fill: 'var(--band)', stroke: 'none' }));
    svg.appendChild(el('path', { d: curve, fill: 'none', stroke: 'var(--series)', 'stroke-width': 2, 'stroke-linejoin': 'round' }));

    // interval bounds + mean markers
    for (const q of [qLo, qHi])
      svg.appendChild(el('line', { x1: X(q), y1: Y(dist.pdf(q)), x2: X(q), y2: M.t + PH, stroke: 'var(--series)', 'stroke-width': 1.5, 'stroke-dasharray': '4 3' }));
    svg.appendChild(el('line', { x1: X(dist.mean), y1: Y(dist.pdf(dist.mean)), x2: X(dist.mean), y2: M.t + PH, stroke: 'var(--muted)', 'stroke-width': 1, 'stroke-dasharray': '2 3' }));

    // direct labels on bounds (merge when the band is narrow on screen)
    const clampX = (x, pad) => Math.min(Math.max(x, M.l + pad), M.l + PW - pad);
    if (X(qHi) - X(qLo) < 110) {
      const lbl = el('text', { x: clampX(X((qLo + qHi) / 2), 55), y: M.t - 8, 'text-anchor': 'middle', 'font-size': 13, 'font-weight': 600, fill: 'var(--ink-2)' });
      lbl.textContent = `${fmtPct(qLo)} – ${fmtPct(qHi)}`;
      svg.appendChild(lbl);
    } else {
      const lo = el('text', { x: X(qLo) - 6, y: M.t - 8, 'text-anchor': 'end', 'font-size': 13, 'font-weight': 600, fill: 'var(--ink-2)' });
      lo.textContent = fmtPct(qLo);
      const hi = el('text', { x: X(qHi) + 6, y: M.t - 8, 'text-anchor': 'start', 'font-size': 13, 'font-weight': 600, fill: 'var(--ink-2)' });
      hi.textContent = fmtPct(qHi);
      svg.appendChild(lo); svg.appendChild(hi);
    }

    // crosshair (populated on hover)
    const cross = el('line', { x1: 0, y1: M.t, x2: 0, y2: M.t + PH, stroke: 'var(--muted)', 'stroke-width': 1, visibility: 'hidden' });
    cross.id = 'cross';
    svg.appendChild(cross);
    const overlay = el('rect', { x: M.l, y: M.t, width: PW, height: PH, fill: 'transparent' });
    overlay.id = 'overlay';
    svg.appendChild(overlay);

    view = { x0, x1, X };

    // readouts
    $('tWidth').textContent = '±' + (half * 100).toFixed(half < 0.005 ? 2 : 1) + ' pp';
    $('tCI').textContent = `${fmtPct(qLo)} – ${fmtPct(qHi)}`;
    $('tCIlvl').textContent = `${Math.round(level * 100)}% credible`;
    $('tMean').textContent = fmtPct(dist.mean);
    $('tBeta').textContent = `${fmtNum(a)}, ${fmtNum(b)}`;
    const cells = $('qtable').tBodies[0].rows[0].cells;
    [0.025, 0.10, 0.25, 0.50, 0.75, 0.90, 0.975].forEach((q, i) => {
      cells[i + 1].textContent = fmtPct(dist.quantile(q), 2);
    });
  }

  // ---------- hover ----------
  svg.addEventListener('pointermove', ev => {
    if (!view) return;
    const r = svg.getBoundingClientRect();
    const sx = (ev.clientX - r.left) / r.width * W;
    if (sx < M.l || sx > M.l + PW) { hideTip(); return; }
    const x = view.x0 + (sx - M.l) / PW * (view.x1 - view.x0);
    const cross = document.getElementById('cross');
    cross.setAttribute('x1', sx); cross.setAttribute('x2', sx);
    cross.setAttribute('visibility', 'visible');
    tip.innerHTML = `CVR <strong>${fmtPct(x)}</strong> · P(θ ≤ ${fmtPct(x, 0)}) = <strong>${fmtPct(dist.cdfAt(x))}</strong>`;
    tip.style.display = 'block';
    const card = svg.parentElement.getBoundingClientRect();
    let left = ev.clientX - card.left + 14;
    if (left + tip.offsetWidth > card.width - 8) left = ev.clientX - card.left - tip.offsetWidth - 14;
    tip.style.left = left + 'px';
    tip.style.top = (ev.clientY - card.top - 34) + 'px';
  });
  function hideTip() {
    tip.style.display = 'none';
    const cross = document.getElementById('cross');
    if (cross) cross.setAttribute('visibility', 'hidden');
  }
  svg.addEventListener('pointerleave', hideTip);

  // ---------- controls ----------
  const bindSlider = (id, out, fmt, set) => {
    $(id).addEventListener('input', ev => {
      const v = +ev.target.value;
      $(out).textContent = fmt(v);
      set(v);
      render();
    });
  };
  bindSlider('p', 'pOut', v => v.toFixed(1) + '%', v => state.p = v / 100);
  bindSlider('n', 'nOut', v => v.toLocaleString('en-US'), v => state.n = v);
  bindSlider('w', 'wOut', v => v.toLocaleString('en-US'), v => state.w = v);
  for (const seg of ['level', 'zoom']) {
    $(seg).addEventListener('click', ev => {
      const btn = ev.target.closest('button');
      if (!btn) return;
      for (const b of $(seg).querySelectorAll('button')) b.setAttribute('aria-pressed', b === btn);
      state[seg] = seg === 'level' ? +btn.dataset.v : btn.dataset.v;
      render();
    });
  }

  render();
})();
</script>
"""

HTML(explorer_html)

Percentile,2.5%,10%,25%,50%,75%,90%,97.5%
CVR,—,—,—,—,—,—,—
